In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import seaborn as sns
from pathlib import Path
import json

# _**File Extraction**_

Two sensor types were used to measure construction noise across sites and time. This file describes the extraction of the data from the Munisense and CityAI Lab sensors.

### CityAI Lab

The data provided by CityAI Lab was structured into two main folders. One follows the "lake" structure, and includes raw, unprocessed data as collected directly by the sensor. The other follows a "marts" structure and is aggregated by minute. Due to technical limitations, processing of raw "lake" data was not possible. Data is thus extracted from the "marts" structure.

The data in this set is split into days, themselves split by sensor, and is extracted by the "load_cityai_data" method.

In [2]:
def load_cityai_data(base_path, dates, sensors):
    dfs = []
    for date in dates:
        for sensor in sensors:
            folder_path = (
                Path(base_path)
                / f"date={date}"
                / f"sensor_id=ics-{sensor}"
            )

            if folder_path.exists():
                parquet_files = list(folder_path.glob("*.parquet"))

                if not parquet_files:
                    print(f"No parquet files found: {folder_path}")
                    continue

                for pq_file in parquet_files:
                    df = pd.read_parquet(pq_file)
                    df["date"] = date
                    df["sensor"] = sensor
                    dfs.append(df)

            else:
                print(f"Missing: {folder_path}")

    merged_df = pd.concat(dfs, ignore_index=True)
    merged_df = merged_df.drop_duplicates(subset=['sensor', 'date', 'time'])

    return merged_df

The method is then applied to the folder, and an initial filter is applied to remove certain sensors.

In [3]:
base_path = r"C:\Users\tangu\PycharmProjects\MSc-Project\Final_Upload_Content\CityAI\loudness_per_minute"

dates = pd.date_range(
    start="2025-06-17",
    end="2025-10-14"
).strftime("%Y-%m-%d")

sensors = [
    "02",
    "03",
    "04",
    "05",
    "06",
    "07",
    "09",
    "11",
    "12"
]

df_CityAI = load_cityai_data(base_path, dates, sensors)
df_CityAI["datetime"] = pd.to_datetime(df_CityAI["time"], format="%Y-%m-%d %H:%M:%S")
df_CityAI["date"] = df_CityAI["datetime"].dt.normalize()
df_CityAI["time"] = df_CityAI["datetime"].dt.time
df_CityAI = df_CityAI[["sensor", "datetime", "date", "time", "dba", "avg_spla"]]
df_CityAI.head()

,sensor,datetime,date,time,dba,avg_spla
0,02,2025-06-17 07:56:00,2025-06-17,07:56:00,45.716390,3.729400e+04
1,02,2025-06-17 00:15:00,2025-06-17,00:15:00,75.274475,3.368585e+07
2,02,2025-06-17 00:16:00,2025-06-17,00:16:00,38.263961,6.704958e+03
3,02,2025-06-17 00:55:00,2025-06-17,00:55:00,36.930865,4.932721e+03
4,02,2025-06-17 01:21:00,2025-06-17,01:21:00,58.527161,7.123872e+05


This code can then be exported. Updated save path for re-use.

In [4]:
df_CityAI.to_csv(r'C:\Users\tangu\PycharmProjects\MSc-Project\Final_Upload_Content\CityAI_data.csv')

### Noise Events

Noise events were measured by CityAI Lab. Analysing them can provide insight into what causes construction noise and associated annoyance and negative health outcomes. This will ultimately help guide problem and objective identification, and design of the framework.

Update path on line 37 for use.

In [5]:
from pathlib import Path
import pandas as pd

def load_cityai_events(base_path, dates, sensors):

    dfs = []

    for date in dates:
        for sensor in sensors:
            folder_path = (
                Path(base_path)
                / f"date={date}"
                / f"sensor_id=ics-{sensor}"
            )

            if folder_path.exists():
                parquet_files = list(folder_path.glob("*.parquet"))

                if not parquet_files:
                    print(f"No parquet files found: {folder_path}")
                    continue

                for pq_file in parquet_files:
                    df = pd.read_parquet(pq_file)
                    df["date"] = date
                    df["sensor"] = sensor
                    dfs.append(df)

            else:
                print(f"Missing: {folder_path}")

    merged_df = pd.concat(dfs, ignore_index=True)

    return merged_df


base_path = r"C:\Users\tangu\PycharmProjects\MSc-Project\Final_Upload_Content\CityAI\scsm_sources"

dates = pd.date_range(
    start="2025-06-17",
    end="2025-10-14"
).strftime("%Y-%m-%d")

sensors = [
    "02", "03", "04", "05", "06",
    "07", "09", "11", "12"
]

df_CityAI_sources = load_cityai_events(base_path, dates, sensors)
df_CityAI_sources.head()

,time,AngleGrinding,Drill,Generator,Human,JackHammer,Miscconstruct,Piling,Reverse,Siren,...,Silent,label,avg_spl_a,max_spl_a,avg_dba,max_dba,avg_sharpness,max_sharpness,date,sensor
0,2025-06-17 01:59:51.623,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,Piling,NaN,NaN,NaN,NaN,NaN,NaN,2025-06-17,02
1,2025-06-17 01:59:54.631,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,Miscconstruct,5.217909e+08,6808.336310,37.356878,38.330410,1.728288,1.234083,2025-06-17,02
2,2025-06-17 01:59:57.646,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,Siren,1.487584e+08,5594.493489,36.411144,37.477608,2.513332,1.356058,2025-06-17,02
3,2025-06-17 02:00:00.654,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,JackHammer,2.797754e+08,17365.715749,36.713660,42.396927,2.772606,1.665136,2025-06-17,02
4,2025-06-17 02:00:03.662,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,Piling,7.176599e+08,4361.296683,35.297154,36.396156,2.363627,1.533018,2025-06-17,02


Update path below for use

In [6]:
df_CityAI_sources.to_csv(r'C:\Users\tangu\PycharmProjects\MSc-Project\Final_Upload_Content\CityAI_sources_data.csv')

### Munisense

We can also extract the data for both sites measured by Munisense sensors.

In [7]:
# Corresponds to the demolition and renovation sites respectively
df_omval = pd.read_csv(r"C:\Users\tangu\PycharmProjects\MSc-Project\Final_Upload_Content\Munisense\omval.csv", sep = ";")
df_merca = pd.read_csv(r"C:\Users\tangu\PycharmProjects\MSc-Project\Final_Upload_Content\Munisense\merca.csv", sep = ";")

In [8]:
df_Munisense = pd.concat([df_omval, df_merca], ignore_index=True)
# Keep and rename columns
df_Munisense = df_Munisense[["#object_id", "result_timestamp", "laeq_avg"]]
df_Munisense = df_Munisense.rename(columns={
    "#object_id": "sensor",
    "result_timestamp": "datetime",
    "laeq_avg": "dba"
})

# Ensures consistancy across networks for the datetime column.
df_Munisense["datetime"] = pd.to_datetime(df_Munisense["datetime"], format="%d-%m-%Y %H:%M:%S")

# Splits into date and time columns
df_Munisense["date"] = df_Munisense["datetime"].dt.normalize()        # YYYY-MM-DD as datetime64
df_Munisense["time"] = df_Munisense["datetime"].dt.time               # HH:MM:SS

# Adds spla for aggregation.
df_Munisense["avg_spla"] = 10 ** (df_Munisense["dba"] / 10)
df_Munisense = df_Munisense[["sensor", "datetime", "date", "time", "dba", "avg_spla"]]

df_Munisense.head()

,sensor,datetime,date,time,dba,avg_spla
0,45,2025-06-13 16:32:00,2025-06-13,16:32:00,83.924269,2.468465e+08
1,45,2025-06-13 16:33:00,2025-06-13,16:33:00,41.074040,1.280572e+04
2,45,2025-06-13 16:34:00,2025-06-13,16:34:00,44.452370,2.787642e+04
3,45,2025-06-13 16:35:00,2025-06-13,16:35:00,88.570955,7.196072e+08
4,45,2025-06-13 16:36:00,2025-06-13,16:36:00,36.590178,4.560556e+03


Munisense ran for longer than CityAI, so for the analysis, a start and end date are defined.

In [9]:
CITYAI_START = "2025-06-17"
CITYAI_END = "2025-10-14 23:59:59"

df_Munisense = df_Munisense[
    (df_Munisense["datetime"] >= CITYAI_START) &
    (df_Munisense["datetime"] <= CITYAI_END)
]

Finally, the file is exported.

In [10]:
df_Munisense.to_csv(r'C:\Users\tangu\PycharmProjects\MSc-Project\Final_Upload_Content\Munisense_data.csv')

### Survey

Survey results were saved as an excel file. They are extracted and pre-processed here.

In [11]:
df_Survey = pd.read_excel(r"C:\Users\tangu\PycharmProjects\MSc-Project\Final_Upload_Content\Survey\Survey_Results_Treated.xlsx", sheet_name="Sheet1")

For ease of analysis, each question is mapped to a simpler label.

In [12]:
rename_map = {
    # Metadata
    "Timestamp": "Timestamp",
    "Consent": "Consent",
    "ZIP": "Please enter your ZIP Code (this survey will be distributed around multiple sites).",

    # Q1: How common are these sound sources
    "Q1_Traffic": "Thinking of the last 12 months, how common were the following sound sources around your home? [Traffic]",
    "Q1_Aviation": "Thinking of the last 12 months, how common were the following sound sources around your home? [Aviation]",
    "Q1_People": "Thinking of the last 12 months, how common were the following sound sources around your home? [People outside talking or screaming]",
    "Q1_Construction": "Thinking of the last 12 months, how common were the following sound sources around your home? [Construction noise]",
    "Q1_Neighbour": "Thinking of the last 12 months, how common were the following sound sources around your home? [Neighbour Noise (TV, door slamming...)]",

    # Q2: How annoying are these sound sources
    "Q2_Traffic": "Thinking of the last 12 months, how annoying were the following sound sources around your home? [Traffic]",
    "Q2_Aviation": "Thinking of the last 12 months, how annoying were the following sound sources around your home? [Aviation]",
    "Q2_People": "Thinking of the last 12 months, how annoying were the following sound sources around your home? [People outside talking or screaming]",
    "Q2_Construction": "Thinking of the last 12 months, how annoying were the following sound sources around your home? [Construction noise]",
    "Q2_Neighbour": "Thinking of the last 12 months, how annoying were the following sound sources around your home? [Neighbour Noise (TV, door slamming...)]",

    # Q3: General noise behavioural effects
    "Q3_WokeUp": "Thinking of the last 12 months, did noise around your home affect your behaviour in the ways described? [Woke me up or kept me from falling asleep]",
    "Q3_ClosedWindows": "Thinking of the last 12 months, did noise around your home affect your behaviour in the ways described? [Caused me to close windows or doors]",
    "Q3_RaisedVoice": "Thinking of the last 12 months, did noise around your home affect your behaviour in the ways described? [Made me raise my voice when speaking]",
    "Q3_Mood": "Thinking of the last 12 months, did noise around your home affect your behaviour in the ways described? [Worsened my mood (irritation, stress, discomfort)]",
    "Q3_Health": "Thinking of the last 12 months, did noise around your home affect your behaviour in the ways described? [Worsened my health (headaches, hearing problems)]",
    "Q3_Distracted": "Thinking of the last 12 months, did noise around your home affect your behaviour in the ways described? [Distracted me from work or studies]",
    "Q3_Report": "Thinking of the last 12 months, did noise around your home affect your behaviour in the ways described? [Made me want to report the noise]",

    # Q4: Construction-specific behavioural effects
    "Q4_WokeUp": "Thinking of the last 12 months, did construction noise around your home affect your behaviour in the ways described? [Woke me up or kept me from falling asleep]",
    "Q4_ClosedWindows": "Thinking of the last 12 months, did construction noise around your home affect your behaviour in the ways described? [Caused me to close windows or doors]",
    "Q4_RaisedVoice": "Thinking of the last 12 months, did construction noise around your home affect your behaviour in the ways described? [Made me raise my voice when speaking]",
    "Q4_Mood": "Thinking of the last 12 months, did construction noise around your home affect your behaviour in the ways described? [Worsened my mood (irritation, stress, discomfort)]",
    "Q4_Health": "Thinking of the last 12 months, did construction noise around your home affect your behaviour in the ways described? [Worsened my health (headaches, hearing problems)]",
    "Q4_Distracted": "Thinking of the last 12 months, did construction noise around your home affect your behaviour in the ways described? [Distracted me from work or studies]",
    "Q4_Report": "Thinking of the last 12 months, did construction noise around your home affect your behaviour in the ways described? [Made me want to report the noise]",

    # Q5: Which characteristics of construction noise annoy you
    "Q5_Volume": "Thinking of the last 12 months, how did the following characteristics of construction noise annoy you? [Volume/loudness of noise]",
    "Q5_Bursts": "Thinking of the last 12 months, how did the following characteristics of construction noise annoy you? [Short bursts of noise]",
    "Q5_LowFreq": "Thinking of the last 12 months, how did the following characteristics of construction noise annoy you? [Low frequency sounds]",
    "Q5_HighFreq": "Thinking of the last 12 months, how did the following characteristics of construction noise annoy you? [High frequency sounds]",
    "Q5_Morning": "Thinking of the last 12 months, how did the following characteristics of construction noise annoy you? [Noise during the morning]",
    "Q5_Afternoon": "Thinking of the last 12 months, how did the following characteristics of construction noise annoy you? [Noise during the afternoon]",
    "Q5_Evening": "Thinking of the last 12 months, how did the following characteristics of construction noise annoy you? [Noise during the evening/night]",

    # Q6: Open text
    "Q6_OpenText": "Please leave any description of your experience with construction noise if you desire. Please do not include any personal information.",
}

# Keeps a lookup for full question text, useful for plot titles later.
question_labels = dict(rename_map)

# Renames columns
df_Survey = df_Survey.rename(columns={full: short for short, full in rename_map.items()})

print(df_Survey.columns.tolist())

['Timestamp', 'Consent', 'ZIP', 'Q1_Traffic', 'Q1_Aviation', 'Q1_People', 'Q1_Construction', 'Q1_Neighbour', 'Q2_Traffic', 'Q2_Aviation', 'Q2_People', 'Q2_Construction', 'Q2_Neighbour', 'Q3_WokeUp', 'Q3_ClosedWindows', 'Q3_RaisedVoice', 'Q3_Mood', 'Q3_Health', 'Q3_Distracted', 'Q3_Report', 'Q4_WokeUp', 'Q4_ClosedWindows', 'Q4_RaisedVoice', 'Q4_Mood', 'Q4_Health', 'Q4_Distracted', 'Q4_Report', 'Q5_Volume', 'Q5_Bursts', 'Q5_LowFreq', 'Q5_HighFreq', 'Q5_Morning', 'Q5_Afternoon', 'Q5_Evening', 'Q6_OpenText']


Residents were asked to provide their postal code. They are mapped to their respective sites here. This was not used in the actual analysis.

In [13]:
postcode_map = {
    "1079LG": "Demolition",
    "1097DS": "Demolition",
    "1096AA": "Demolition",
    "1096HV": "Demolition",
    "1056CL": "Infrastructure",
    "1057BV": "Infrastructure",
    "1091EX": "Construction",
    "1091GA": "Construction",
    "1091EZ": "Construction"
}

postcode_col = "ZIP"

# Normalises postcodes in case of uneven writing.
normalised_postcodes = df_Survey[postcode_col].astype(str).str.upper().str.replace(" ", "", regex=False)

# Maps to site name, anything not in the dictionary becomes NaN values.
site_column = normalised_postcodes.map(postcode_map)
df_Survey.insert(3, "site", site_column)

print(df_Survey["site"].value_counts(dropna=False))


site
Demolition        22
Construction       7
Infrastructure     2
Name: count, dtype: int64


The survey data is finally exported, along with the mapping in json format.

In [14]:
with open("question_labels.json", "w") as f:
    json.dump(question_labels, f, indent=2)

df_Survey.to_csv(r'C:\Users\tangu\PycharmProjects\MSc-Project\Final_Upload_Content\Survey_data.csv')